# Google Colab Bootstrap

**Brain Tumour Classification from Magnetic Resonance Imaging**

Course: DS 4889 Machine Learning, United International University

---

## Purpose

This notebook prepares a Google Colab session for training. It is not part of
the scientific analysis; notebooks 01 to 05 contain that work. Its role is to
obtain a graphics processor, install dependencies, download the dataset and
verify that the pipeline executes before a long training run is committed to.

## Why train on Colab rather than locally

TensorFlow releases after version 2.10 do not provide native graphics
processor support on Windows. Training on a central processor takes
approximately four to seven hours per model, and this study trains two models.
Colab provides a free graphics processor that reduces this to roughly twenty
minutes per model.

A reasonable division of labour is therefore to train on Colab, download the
resulting model artefact, and run the Streamlit application locally.

## Before starting

Open `Runtime`, then `Change runtime type`, and set the hardware accelerator to
`T4 GPU`. The first cell will report whether this succeeded.

## Step 1: Confirm the hardware accelerator

If no graphics processor is reported, the remaining cells will still execute but
training will be impractically slow. Change the runtime type before continuing.

In [ ]:
import subprocess
import sys

try:
    import tensorflow as tf

    devices = tf.config.list_physical_devices('GPU')
    print('TensorFlow version:', tf.__version__)
    print('Graphics processors visible:', len(devices))
    for device in devices:
        print('   ', device.name)
    if not devices:
        print()
        print('No graphics processor detected.')
        print('Open Runtime, then Change runtime type, and select T4 GPU.')
except ImportError:
    print('TensorFlow is not yet available in this session.')

print()
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'nvidia-smi unavailable')

## Step 2: Obtain the project source

Two routes are available. Use the repository route once the project has been
pushed to GitHub, because it guarantees that the session runs the committed
version. Use the upload route for a quick trial before publishing.

Set `REPOSITORY_URL` to your repository address, or leave it as `None` to upload
a zip archive of the project folder instead.

In [ ]:
import os
from pathlib import Path

REPOSITORY_URL = 'https://github.com/nafiul-afk/gmm-cnn-brain-tumor.git'
PROJECT_NAME = 'gmm-cnn-brain-tumor'

if REPOSITORY_URL:
    if not Path(PROJECT_NAME).exists():
        os.system(f'git clone {REPOSITORY_URL} {PROJECT_NAME}')
    os.chdir(PROJECT_NAME)
    print('Working directory:', os.getcwd())
else:
    print('No repository configured.')
    print('Compress the project folder locally, then run the next cell to upload it.')

In [ ]:
# Upload route. Skip this cell if the repository was cloned above.
import os
import zipfile
from pathlib import Path

if not Path('src').exists():
    from google.colab import files

    print('Select a zip archive of the project folder.')
    uploaded = files.upload()

    for filename in uploaded:
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as archive:
                archive.extractall('.')
            print('Extracted', filename)

    # The archive may contain a single top level folder.
    if not Path('src').exists():
        candidates = [p for p in Path('.').iterdir() if p.is_dir() and (p / 'src').exists()]
        if candidates:
            os.chdir(candidates[0])

print('Working directory:', os.getcwd())
print('Source package present:', Path('src').exists())

## Step 3: Install dependencies

Colab preinstalls most of the required libraries. Only the packages absent from
the default image are installed here, which keeps session startup short.

The Streamlit dependency is not required for training and is omitted.

In [ ]:
import subprocess
import sys

REQUIRED = ['statsmodels', 'tqdm', 'opencv-python-headless', 'seaborn', 'plotly']

missing = []
IMPORT_NAMES = {
    'opencv-python-headless': 'cv2',
    'statsmodels': 'statsmodels',
    'tqdm': 'tqdm',
    'seaborn': 'seaborn',
    'plotly': 'plotly',
}

for package in REQUIRED:
    try:
        __import__(IMPORT_NAMES[package])
    except ImportError:
        missing.append(package)

if missing:
    print('Installing:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *missing], check=False)
else:
    print('All required packages are already present.')

print('Dependency check complete.')

## Step 4: Authenticate with Kaggle

The dataset is distributed through Kaggle and requires an API token.

To obtain one, sign in at `kaggle.com`, open `Settings`, scroll to the API
section and select `Create New Token`. A file named `kaggle.json` is downloaded.
Upload that file when prompted by the next cell.

The token is a credential. Do not commit it to version control and do not share
the notebook with its output retained after running this cell.

In [ ]:
import os
import shutil
from pathlib import Path

token_path = Path.home() / '.kaggle' / 'kaggle.json'

if token_path.exists():
    print('Kaggle credentials already configured.')
else:
    from google.colab import files

    print('Select kaggle.json.')
    uploaded = files.upload()

    token_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move('kaggle.json', token_path)
    os.chmod(token_path, 0o600)
    print('Credentials installed at', token_path)

## Step 5: Download and extract the dataset

The archive is approximately 150 megabytes. The extraction step checks for the
common failure in which the content lands inside an additional nested folder,
and corrects it.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

if (DATA_DIR / 'Training').exists():
    print('Dataset already present.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'kaggle'], check=False)
    subprocess.run(
        [
            'kaggle', 'datasets', 'download',
            '-d', 'masoudnickparvar/brain-tumor-mri-dataset',
            '-p', str(DATA_DIR),
        ],
        check=False,
    )

    archives = list(DATA_DIR.glob('*.zip'))
    for archive_path in archives:
        with zipfile.ZipFile(archive_path, 'r') as archive:
            archive.extractall(DATA_DIR)
        archive_path.unlink()
        print('Extracted', archive_path.name)

    # Correct a nested extraction if one occurred.
    if not (DATA_DIR / 'Training').exists():
        for child in DATA_DIR.iterdir():
            if child.is_dir() and (child / 'Training').exists():
                for item in child.iterdir():
                    shutil.move(str(item), str(DATA_DIR / item.name))
                child.rmdir()
                print('Corrected nested extraction from', child.name)
                break

print('Training folder present:', (DATA_DIR / 'Training').exists())
print('Testing folder present: ', (DATA_DIR / 'Testing').exists())

## Step 6: Verify the dataset layout

Every row must report a non zero image count. A count of zero indicates that the
archive extracted into an unexpected structure, which must be corrected before
training.

In [ ]:
import sys

if '.' not in sys.path:
    sys.path.insert(0, '.')

from src import data_loader

inventory = data_loader.verify_dataset()
print(inventory.to_string(index=False))
print()
print('Total images:', int(inventory['image_count'].sum()))

if inventory['image_count'].min() == 0:
    print()
    print('At least one class folder is empty. Resolve this before training.')
else:
    print('Dataset layout verified.')

## Step 7: Smoke test

A short run on a small subset confirms that the pipeline executes end to end
before a multi hour training run is started. Three epochs on one hundred images
per class complete in a few minutes.

The accuracy produced here is not meaningful and should not be reported. The
purpose is solely to surface runtime errors.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable, '-m', 'src.train',
        '--experiments', 'baseline',
        '--epochs', '3',
        '--limit-per-class', '100',
    ],
    capture_output=True,
    text=True,
)

print(result.stdout[-4000:])
if result.returncode != 0:
    print('STANDARD ERROR')
    print(result.stderr[-4000:])
else:
    print()
    print('Smoke test completed successfully.')

## Step 8: Full training run

This executes the complete experimental protocol: the baseline convolutional
network and the hybrid pipeline, under identical seeds, partitions and
schedules.

Expect approximately fifty minutes in total on a T4 graphics processor. The
segmentation stage runs on the central processor and accounts for roughly half
of that time; its output is cached, so a repeated run is substantially faster.

Colab disconnects idle sessions. Keep the browser tab active while this runs.

In [ ]:
import subprocess
import sys
import time

start = time.time()

process = subprocess.Popen(
    [
        sys.executable, '-m', 'src.train',
        '--experiments', 'baseline', 'hybrid',
        '--epochs', '50',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='')

process.wait()
print()
print('Elapsed:', round((time.time() - start) / 60, 1), 'minutes')
print('Exit code:', process.returncode)

## Step 9: Review the results

The comparison table and the paired significance test are written to the results
directory by the training script.

In [ ]:
import json
from pathlib import Path

import pandas as pd

comparison_path = Path('results/metrics/experiment_comparison.csv')
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    print('Model comparison')
    print(comparison.to_string(index=False))
else:
    print('No comparison table found. Confirm that training completed.')

mcnemar_path = Path('results/metrics/mcnemar_test.json')
if mcnemar_path.exists():
    print()
    print('Paired significance test')
    with mcnemar_path.open('r', encoding='utf-8') as handle:
        print(json.dumps(json.load(handle), indent=2))

## Step 10: Retrieve the artefacts

Download the trained model and the generated results so that the Streamlit
application can be run locally and the figures are available for the project
write up.

Place the downloaded model in the `models` directory of the local project.

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

best_model = Path('models/best_model.keras')
if best_model.exists():
    size_mb = round(best_model.stat().st_size / (1024 * 1024), 1)
    print('Downloading best_model.keras,', size_mb, 'megabytes')
    files.download(str(best_model))
else:
    print('No model artefact found.')

results_dir = Path('results')
if results_dir.exists() and any(results_dir.rglob('*')):
    shutil.make_archive('results_bundle', 'zip', results_dir)
    print('Downloading results_bundle.zip')
    files.download('results_bundle.zip')

## Optional: persist artefacts to Google Drive

Colab sessions are ephemeral. Mounting Drive and copying the artefacts there
protects against losing a completed training run to a disconnection.

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

destination = Path('/content/drive/MyDrive/brain_tumour_project')
destination.mkdir(parents=True, exist_ok=True)

for folder in ('models', 'results'):
    source = Path(folder)
    if source.exists():
        shutil.copytree(source, destination / folder, dirs_exist_ok=True)
        print('Copied', folder, 'to', destination / folder)

## Next steps

With training complete and the artefacts retrieved:

1. Place `best_model.keras` in the `models` directory of the local project.
2. Run the application locally with `streamlit run app/streamlit_app.py`.
3. Execute notebooks 01 to 05 to produce the full set of figures and tables.
   Notebooks 03 and 04 will reuse the cached artefacts rather than retraining,
   provided the cache was retrieved alongside the model.
4. Update the results section of the README from the computed values in
   `results/metrics`.

Report the outcome of the paired significance test in whichever direction it
falls. A result showing no significant improvement is a valid finding and should
be stated plainly rather than omitted.